In [ ]:
import os
from pathlib import Path
import shutil

import teehr

teehr.__version__

### Setup

In [ ]:
test_eval_dir = Path(Path().cwd(), "FIRO_test_evaluation")

# spark configuration
from teehr.evaluation.spark_session_utils import create_spark_session
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=2,
    executor_memory="50g",
    executor_cores=7
)

# access local evaluation
ev = teehr.LocalReadWriteEvaluation(
    dir_path=test_eval_dir,
    spark=spark
)

ev.enable_logging()

### Assemble JTS

In [ ]:
# Add RCFs useful for grouping

from teehr import RowLevelCalculatedFields as rcf

ev.joined_timeseries_view().add_calculated_fields([
    rcf.Month(),
    rcf.Year(),
    rcf.WaterYear(),
    rcf.Seasons(),
    rcf.ForecastLeadTime(),
    rcf.DayOfYear(),
]).write(
    "joined_timeseries"
)

In [ ]:
# define custom uniqueness_fields arg so we obtain a deterministic quantile as opposed to several quantiles per reference_time
custom_uniqueness_fields = [
    'primary_location_id',
    'configuration_name',
    'variable_name',
    'unit_name'
]

# Add quantiles useful for threshold metrics
q_10th = teehr.TimeseriesAwareCalculatedFields.AbovePercentileEventDetection(
    quantile=0.1,
    output_event_field_name='event_10th',
    output_event_id_field_name='event_10th_id',
    output_quantile_field_name='q_10th',
    add_quantile_field=True,
    uniqueness_fields=custom_uniqueness_fields
)

q_25th = teehr.TimeseriesAwareCalculatedFields.AbovePercentileEventDetection(
    quantile=0.25,
    output_event_field_name='event_25th',
    output_event_id_field_name='event_25th_id',
    output_quantile_field_name='q_25th',
    add_quantile_field=True,
    uniqueness_fields=custom_uniqueness_fields
)

q_50th = teehr.TimeseriesAwareCalculatedFields.AbovePercentileEventDetection(
    quantile=0.5,
    output_event_field_name='event_50th',
    output_event_id_field_name='event_50th_id',
    output_quantile_field_name='q_50th',
    add_quantile_field=True,
    uniqueness_fields=custom_uniqueness_fields
)

q_75th = teehr.TimeseriesAwareCalculatedFields.AbovePercentileEventDetection(
    quantile=0.75,
    output_event_field_name='event_75th',
    output_event_id_field_name='event_75th_id',
    output_quantile_field_name='q_75th',
    add_quantile_field=True,
    uniqueness_fields=custom_uniqueness_fields
)

q_90th = teehr.TimeseriesAwareCalculatedFields.AbovePercentileEventDetection(
    quantile=0.9,
    output_event_field_name='event_90th',
    output_event_id_field_name='event_90th_id',
    output_quantile_field_name='q_90th',
    add_quantile_field=True,
    uniqueness_fields=custom_uniqueness_fields
)

# write to eval
ev.table("joined_timeseries").add_calculated_fields([
    q_10th,
    q_25th,
    q_50th,
    q_75th,
    q_90th,
]).write(
    "joined_timeseries"
)

In [ ]:
# ADD DAILY FORECAST LEAD TIME BINS OR SOME CUSTOM IMPLEMENTATION

fcst_bins = rcf.ForecastLeadTimeBins(
    bin_size='1 day'
)
ev.table("joined_timeseries").add_calculated_fields([
    fcst_bins,
]).write(
    "joined_timeseries"
)

### Kill Spark

In [ ]:
ev.spark.stop()